# e2 · Macro value in a linear model (block B2)

Do macro variables, including VIX, improve forecasts over past log RV alone?
Compare `ols_1m` (OLS), `en_no_macro` (Elastic Net with the same single predictor),
and `en_full_macro` (Elastic Net adding all 11 macro variables). The three pairs
are defined locally; `research_config.BLOCKS["B2"]` is unchanged.

**Tests.** DM compares raw forecast losses for log MSE and QLIKE. The two macro
pairs also report Clark-West (CW) for log MSE. Under nested-model assumptions,
CW adjusts for estimating extra predictors; its significance is not significance
of the observed raw gain. Predictor sets are nested here, but separately tuned
Elastic Net estimation makes CW exploratory. DM p-values also require caution
for nested comparisons.

Pooled tests average differentials across ETFs each month and use Newey-West
standard errors with Andrews bandwidth. All p-values are one-sided: positive
`gain_t` favours `a` under the corresponding test. `loss_ratio` is always the
unadjusted loss of `a` divided by that of `b`; below 1 favours `a`.

**Output:** `forecasting/results/e2_losses.csv` and `e2_dm_tests.csv`.
Diagnostics are displayed only in this notebook.


In [22]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

MODELS_DIR = next(
    candidate
    for p in [Path.cwd(), *Path.cwd().parents]
    for candidate in [p, p / "models"]
    if (candidate / "research_config.py").is_file()
)
sys.path.insert(0, str(MODELS_DIR))
from research_config import TICKERS

FORECAST_DIR = MODELS_DIR / "forecasting" / "forecasts"
RESULTS_DIR = MODELS_DIR / "forecasting" / "results"
RESULTS_DIR.mkdir(exist_ok=True)

models = ["ols_1m", "en_no_macro", "en_full_macro"]
# Pair ID, a, b, what the comparison isolates.
pairs = [
    ("B2.1", "en_full_macro", "ols_1m", "total effect of macro + VIX and estimation"),
    ("B2.2", "en_full_macro", "en_no_macro", "macro + VIX within Elastic Net"),
    ("B2.3", "en_no_macro", "ols_1m", "regularization with the same predictor"),
]
# The OLS builder currently writes to the root forecasts directory. Refuse stale copies.
OLS_PATH = FORECAST_DIR / "ols_forecasts_expanding_tidy.csv"
OLS_BUILDER_PATH = MODELS_DIR.parent / "forecasts" / "ols_forecasts_expanding_tidy.csv"
assert OLS_PATH.is_file() and OLS_BUILDER_PATH.is_file(), "Both OLS forecast copies must exist."
assert OLS_PATH.read_bytes() == OLS_BUILDER_PATH.read_bytes(), (
    "OLS forecast copies differ: synchronize forecasts/ols_forecasts_expanding_tidy.csv "
    "and models/forecasting/forecasts/ols_forecasts_expanding_tidy.csv before evaluation."
)
ols = pd.read_csv(OLS_PATH, parse_dates=["date", "target_date"])
en = pd.read_csv(FORECAST_DIR / "en_forecasts.csv", parse_dates=["date", "target_date"])
df = pd.concat([
    ols[ols["model"].eq("ols_1m")],
    en[en["model"].isin(["en_no_macro", "en_full_macro"])],
], ignore_index=True)
df = df[df["etf"].isin(TICKERS)].copy()
missing = set(models) - set(df["model"])
if missing:
    raise ValueError(f"Missing required models: {sorted(missing)}")
if df.duplicated(["model", "etf", "date"]).any():
    raise ValueError("Duplicate model/ETF/forecast-month rows.")
if df[["date", "target_date"]].isna().any().any():
    raise ValueError("Forecast and target dates must be present.")
print(f"{len(pairs)} B2 pairs runnable")


3 B2 pairs runnable


In [23]:
def andrews_bandwidth(x):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))


def newey_west_lrv(x, lag):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    lag = max(0, min(lag, n - 1)); s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)


def hac_t(d):
    """t-statistic of mean(d) with a Newey-West variance."""
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))


def qlike(rv, fc):
    r = rv / fc
    return r - np.log(r) - 1

## Losses

Common sample: retain only (ETF, forecast month) rows with forecasts from all three models.
The target month and realized outcomes must agree across models. All losses and tests use
this same sample. `date` is the forecast origin; `target_date` is the month being predicted.


In [24]:
d = df[df.groupby(["etf", "date"])["model"].transform("nunique") == len(models)].copy()
d = d.sort_values(["date", "etf", "model"]).reset_index(drop=True)
if d.empty:
    raise ValueError("No common forecast sample across the three models.")
values = ["actual_logRV", "actual_RV", "fc_log", "fc_level"]
if not np.isfinite(d[values].to_numpy()).all():
    raise ValueError("The common sample contains non-finite forecasts or outcomes.")
if (d[["actual_RV", "fc_level"]] <= 0).any().any():
    raise ValueError("QLIKE requires positive realized and forecast variances.")
keys = ["etf", "date"]
if (d.groupby(keys)["target_date"].nunique() != 1).any():
    raise ValueError("Target months differ across models.")
for col in ["actual_logRV", "actual_RV"]:
    wide = d.pivot(index=keys, columns="model", values=col)
    if not np.allclose(wide.to_numpy(), wide[[models[0]]].to_numpy(), rtol=1e-10, atol=1e-12):
        raise ValueError(f"Realized outcomes differ across models: {col}")
# Keep the pooled monthly average equally weighted across the five ETFs.
if (d.groupby("date")["etf"].nunique() != len(TICKERS)).any():
    raise ValueError("The common sample must contain all five ETFs each month.")

d["MSE_log"] = (d["actual_logRV"] - d["fc_log"]) ** 2
d["MSE_level"] = (d["actual_RV"] - d["fc_level"]) ** 2
d["QLIKE"] = qlike(d["actual_RV"], d["fc_level"])
print(f"Forecast origins: {d['date'].min():%Y-%m} to {d['date'].max():%Y-%m}")
print(f"Target months: {d['target_date'].min():%Y-%m} to {d['target_date'].max():%Y-%m}")
print(f"{d['date'].nunique()} months × {len(TICKERS)} ETFs × {len(models)} models = {len(d)} rows")

LOSSES = ["QLIKE", "MSE_log", "MSE_level"]
per_etf = d.groupby(["model", "etf"])[LOSSES].mean().reset_index()
pooled = d.groupby("model")[LOSSES].mean().reset_index().assign(etf="pooled")
losses = pd.concat([per_etf, pooled], ignore_index=True).rename(columns={"etf": "ETF"})
losses.to_csv(RESULTS_DIR / "e2_losses.csv", index=False)
losses[losses["ETF"] == "pooled"].set_index("model")[LOSSES].round(4)


Forecast origins: 2008-02 to 2025-11
Target months: 2008-03 to 2025-12
214 months × 5 ETFs × 3 models = 3210 rows


,QLIKE,MSE_log,MSE_level
model,,,
en_full_macro,0.3247,0.4902,0.0011
en_no_macro,0.3266,0.5228,0.0001
ols_1m,0.3266,0.5229,0.0001


## Tests

- `B2.1`: full-macro Elastic Net vs OLS — macro information plus estimation changes.
- `B2.2`: full-macro vs no-macro Elastic Net — macro information within one family.
- `B2.3`: no-macro Elastic Net vs OLS — regularization with the same predictor; DM only.

Level MSE is descriptive; formal tests cover log MSE and QLIKE.


In [25]:
def one_sided_p(t):
    return float(norm.sf(t))


loss_idx = losses.set_index(["model", "ETF"])
rows = []
cw_components = []
for pair_id, a, b, _ in pairs:
    A = d[d["model"] == a].set_index(["date", "etf"]).sort_index()
    B = d[d["model"] == b].set_index(["date", "etf"]).sort_index()
    dm_diff = A["QLIKE"] - B["QLIKE"]   # negative favours a
    mse_diff = A["MSE_log"] - B["MSE_log"]
    comparisons = [("QLIKE", "DM", dm_diff), ("MSE_log", "DM", mse_diff)]
    if a == "en_full_macro":
        # Clark-West adjusted differential; positive favours a.
        cw = B["MSE_log"] - (A["MSE_log"] - (B["fc_log"] - A["fc_log"]) ** 2)
        comparisons.append(("MSE_log", "Clark-West", cw))
        # Retain the pooled decomposition for diagnostics; do not re-run these tests.
        adjustment = (B["fc_log"] - A["fc_log"]) ** 2
        cw_components.append({
            "pair": pair_id,
            "mean_raw_gain": (-mse_diff).groupby("date").mean().mean(),
            "mean_adjustment_term": adjustment.groupby("date").mean().mean(),
            "mean_CW_differential": cw.groupby("date").mean().mean(),
        })

    for etf in ["pooled"] + TICKERS:
        for loss, test, diff in comparisons:
            series = diff.groupby("date").mean() if etf == "pooled" else diff.xs(etf, level="etf")
            series = series.sort_index()
            if len(series) < 2:
                raise ValueError(f"Insufficient observations for {pair_id}, {etf}.")
            t = -hac_t(series) if test == "DM" else hac_t(series)
            rows.append({
                "pair": pair_id, "a": a, "b": b, "ETF": etf, "loss": loss, "test": test,
                "loss_ratio": loss_idx.loc[(a, etf), loss] / loss_idx.loc[(b, etf), loss],
                "gain_t": t, "p_value": one_sided_p(t),
            })

tests = pd.DataFrame(rows)
tests.to_csv(RESULTS_DIR / "e2_dm_tests.csv", index=False)
tests[tests["ETF"] == "pooled"].drop(columns="ETF").round(4)


,pair,a,b,loss,test,loss_ratio,gain_t,p_value
0,B2.1,en_full_macro,ols_1m,QLIKE,DM,0.9941,0.0848,0.4662
1,B2.1,en_full_macro,ols_1m,MSE_log,DM,0.9374,0.9555,0.1697
2,B2.1,en_full_macro,ols_1m,MSE_log,Clark-West,0.9374,5.9293,0.0000
18,B2.2,en_full_macro,en_no_macro,QLIKE,DM,0.9941,0.0853,0.4660
19,B2.2,en_full_macro,en_no_macro,MSE_log,DM,0.9376,0.9533,0.1702
20,B2.2,en_full_macro,en_no_macro,MSE_log,Clark-West,0.9376,5.9284,0.0000
36,B2.3,en_no_macro,ols_1m,QLIKE,DM,1.0000,-0.1220,0.5486
37,B2.3,en_no_macro,ols_1m,MSE_log,DM,0.9998,1.7925,0.0365


## By sector

Each cell shows `loss_ratio (p-value)` for a sector or the monthly pooled result.
A ratio below 1 favours `a`; p-values are one-sided. Log-MSE DM tests the raw loss
difference, while Clark-West tests the adjusted differential. Both show the same raw
loss ratio. Clark-West is reported only for the two macro-expansion pairs.

In [26]:
SECTORS = {
    "XLB": "Materials", "XLE": "Energy", "XLF": "Financials",
    "XLI": "Industrials", "XLY": "Cons. Disc.",
}


def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def by_sector(loss, test):
    t = tests[tests["loss"].eq(loss) & tests["test"].eq(test)].copy()
    t["cell"] = [f"{ratio:.3f} ({fmt_p(p)})" for ratio, p in zip(t["loss_ratio"], t["p_value"])]
    t["row"] = t["pair"] + ": " + t["a"] + " vs " + t["b"]
    table = t.pivot(index="row", columns="ETF", values="cell").reindex(columns=TICKERS + ["pooled"])
    table.columns = [f"{ticker} {SECTORS[ticker]}" for ticker in TICKERS] + ["Pooled"]
    return table.rename_axis(None)


print("Log-RV MSE: ratio (p-value), DM")
by_sector("MSE_log", "DM")

Log-RV MSE: ratio (p-value), DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
B2.1: en_full_macro vs ols_1m,0.910 (0.082),1.038 (0.696),0.922 (0.104),0.860 (0.054),0.974 (0.370),0.937 (0.170)
B2.2: en_full_macro vs en_no_macro,0.910 (0.082),1.038 (0.696),0.923 (0.105),0.861 (0.055),0.974 (0.371),0.938 (0.170)
B2.3: en_no_macro vs ols_1m,1.000 (0.077),1.000 (0.352),1.000 (0.005),1.000 (0.056),1.000 (0.135),1.000 (0.037)


In [27]:
print("Log-RV MSE: ratio (p-value), Clark-West")
by_sector("MSE_log", "Clark-West")

Log-RV MSE: ratio (p-value), Clark-West


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
B2.1: en_full_macro vs ols_1m,0.910 (<0.001),1.038 (0.269),0.922 (<0.001),0.860 (<0.001),0.974 (<0.001),0.937 (<0.001)
B2.2: en_full_macro vs en_no_macro,0.910 (<0.001),1.038 (0.271),0.923 (<0.001),0.861 (<0.001),0.974 (<0.001),0.938 (<0.001)


In [28]:
print("QLIKE: ratio (p-value), DM")
by_sector("QLIKE", "DM")

QLIKE: ratio (p-value), DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
B2.1: en_full_macro vs ols_1m,0.944 (0.229),1.206 (0.873),0.972 (0.323),0.886 (0.019),0.986 (0.394),0.994 (0.466)
B2.2: en_full_macro vs en_no_macro,0.944 (0.228),1.206 (0.873),0.972 (0.324),0.886 (0.019),0.986 (0.394),0.994 (0.466)
B2.3: en_no_macro vs ols_1m,1.000 (0.619),1.000 (0.780),1.000 (0.371),1.000 (0.623),1.000 (0.385),1.000 (0.549)


## Diagnostics: Clark-West decomposition

For each macro pair, the pooled mean CW differential equals the mean raw gain
`MSE_b - MSE_a` plus the mean adjustment `(fc_log_b - fc_log_a)²`.
This separates the observed gain from the adjustment behind the CW statistic.


In [29]:
cw_decomposition = pd.DataFrame(cw_components).set_index("pair")
assert np.allclose(
    cw_decomposition.mean_CW_differential,
    cw_decomposition.mean_raw_gain + cw_decomposition.mean_adjustment_term,
)
cw_decomposition.round(6)


,mean_raw_gain,mean_adjustment_term,mean_CW_differential
pair,,,
B2.1,0.032707,0.107121,0.139828
B2.2,0.032598,0.107096,0.139694


## Diagnostics: largest variance-level errors

These are the five largest ETF-month errors, not five distinct months. `level_ratio`
is forecast variance divided by realized variance. QLIKE depends on this ratio, whereas
level MSE squares the absolute variance error; extreme over-forecasts can therefore
have very different effects on the two losses. Error deletion below is a descriptive
sensitivity check chosen after seeing outcomes, not a replacement evaluation sample.

In [30]:
macro_errors = d[d.model.eq("en_full_macro")].copy()
macro_errors["level_ratio"] = macro_errors.fc_level / macro_errors.actual_RV
macro_errors["level_error_share"] = macro_errors.MSE_level / macro_errors.MSE_level.sum()
top5 = macro_errors.nlargest(5, "MSE_level")
error_cols = ["etf", "date", "target_date", "actual_RV", "fc_log", "sigma2", "fc_level",
              "level_ratio", "MSE_log", "MSE_level", "QLIKE", "level_error_share"]
print(f"Top five errors account for {top5.level_error_share.sum():.1%} of full-macro level MSE.")
worst_keys = pd.MultiIndex.from_frame(top5[["etf", "date"]])
without_top5 = d[~pd.MultiIndex.from_frame(d[["etf", "date"]]).isin(worst_keys)]
print("Level MSE after removing the same five ETF-months from every model:")
print(without_top5.groupby("model").MSE_level.mean().to_string())
top5[error_cols]

Top five errors account for 81.5% of full-macro level MSE.
Level MSE after removing the same five ETF-months from every model:
model
en_full_macro    0.000200
en_no_macro      0.000104
ols_1m           0.000104


,etf,date,target_date,actual_RV,fc_log,sigma2,fc_level,level_ratio,MSE_log,MSE_level,QLIKE,level_error_share
132,XLY,2008-10-31,2008-11-30,0.045574,-0.528812,0.329611,0.694887,15.247411,6.551574,0.421607,1.789995,0.366374
126,XLF,2008-10-31,2008-11-30,0.087358,-0.929827,0.714462,0.564059,6.456855,2.273796,0.227244,1.020017,0.197474
129,XLI,2008-10-31,2008-11-30,0.024007,-1.069012,0.314917,0.401899,16.741049,7.077757,0.142803,1.877597,0.124095
201,XLF,2009-03-31,2009-04-30,0.062968,-1.343476,0.751981,0.380037,6.035425,2.021106,0.100533,0.963335,0.087363
120,XLB,2008-10-31,2008-11-30,0.035719,-1.584606,0.385108,0.248565,6.958862,3.053623,0.045303,1.083718,0.039368


## Diagnostics: regimes, bias and variance correction

Regimes below are descriptive target-year buckets, not estimated break dates. Bias is
mean forecast minus actual log RV, with `MSE_log = bias² + centered_error_variance`.
The uncorrected exponential is a sensitivity check, not an alternative unbiased forecast:
`exp(fc_log + sigma2/2)` relies on a conditional lognormal approximation, and matching
that formula does not prove that the correction is optimally calibrated.

In [31]:
diagnostics = d.copy()
diagnostics["regime"] = np.select(
    [diagnostics.target_date.dt.year.le(2009), diagnostics.target_date.dt.year.le(2019),
     diagnostics.target_date.dt.year.eq(2020)],
    ["2008-2009", "2010-2019", "2020"], default="2021-2025",
)
diagnostics["log_error"] = diagnostics.fc_log - diagnostics.actual_logRV
regime_rows = []
for (regime, model), g in diagnostics.groupby(["regime", "model"]):
    bias = g.log_error.mean()
    regime_rows.append(dict(regime=regime, model=model, n=len(g), bias_log=bias,
                            bias_squared=bias ** 2, centered_error_variance=g.log_error.var(ddof=0),
                            MSE_log=g.MSE_log.mean(), QLIKE=g.QLIKE.mean(), MSE_level=g.MSE_level.mean()))
regimes = pd.DataFrame(regime_rows)
correction_rows = []
for model, g in diagnostics.groupby("model"):
    error = np.max(np.abs(np.log(g.fc_level) - g.fc_log - 0.5 * g.sigma2))
    assert error < 1e-10, f"Incorrect variance retransformation: {model}"
    for version, fc in [("saved correction", g.fc_level), ("exp only", np.exp(g.fc_log))]:
        correction_rows.append(dict(model=model, version=version, QLIKE=qlike(g.actual_RV, fc).mean(),
                                    MSE_level=((g.actual_RV - fc) ** 2).mean(), formula_error=error))
correction = pd.DataFrame(correction_rows)
print("Overall log-error bias and centered variance:")
print(diagnostics.groupby("model").log_error.agg(["mean", lambda x: x.var(ddof=0)]).to_string())
print("Retransformation sensitivity:")
print(correction.to_string(index=False))
regimes.round(6)

Overall log-error bias and centered variance:
                   mean  <lambda_0>
model                              
en_full_macro  0.034847    0.488943
en_no_macro    0.004636    0.522734
ols_1m         0.004665    0.522843
Retransformation sensitivity:
        model          version    QLIKE  MSE_level  formula_error
en_full_macro saved correction 0.324691   0.001075   1.060263e-13
en_full_macro         exp only 0.363764   0.000663   1.060263e-13
  en_no_macro saved correction 0.326621   0.000107   1.122435e-13
  en_no_macro         exp only 0.376835   0.000103   1.122435e-13
       ols_1m saved correction 0.326613   0.000107   1.215694e-13
       ols_1m         exp only 0.376839   0.000103   1.215694e-13


,regime,model,n,bias_log,bias_squared,centered_error_variance,MSE_log,QLIKE,MSE_level
0,2008-2009,en_full_macro,110,-0.036906,0.001362,0.828495,0.829857,0.582348,0.009583
1,2008-2009,en_no_macro,110,-0.230311,0.053043,0.512869,0.565912,0.390377,0.000455
2,2008-2009,ols_1m,110,-0.229070,0.052473,0.512949,0.565423,0.389928,0.000455
3,2010-2019,en_full_macro,600,0.047019,0.002211,0.451714,0.453925,0.260762,0.000011
4,2010-2019,en_no_macro,600,0.089896,0.008081,0.522956,0.531037,0.293201,0.000011
5,2010-2019,ols_1m,600,0.089630,0.008034,0.523229,0.531262,0.293320,0.000011
6,2020,en_full_macro,60,-0.250800,0.062901,1.298042,1.360943,1.222073,0.001443
7,2020,en_no_macro,60,-0.330504,0.109233,1.205372,1.314605,1.248882,0.000910
8,2020,ols_1m,60,-0.329820,0.108781,1.206004,1.314785,1.248559,0.000911
9,2021-2025,en_full_macro,300,0.093941,0.008825,0.255085,0.263910,0.178599,0.000011


## Diagnostics: independent OLS reconstruction and constant benchmark

Re-fit each `ols_1m` forecast independently using least squares on the first `i` rows.
At origin month t, training targets end at t; the forecast target is t+1. The audit also
recomputes the five-fold time-series out-of-fold residual variance. A constant benchmark
uses only the expanding mean of training log RV, and the random walk predicts current
log RV. Neither uses future targets. The five ETFs are evaluated on the same dates as
above. This validates the saved OLS forecasts, not every macro data release vintage.

In [32]:
from sklearn.model_selection import TimeSeriesSplit

rv_panel = pd.read_csv(MODELS_DIR.parent / "dataset/price/clean/realized_variance.csv", parse_dates=["date"])
audit_rows = []
benchmark_rows = []
for ticker in TICKERS:
    history = rv_panel[rv_panel.etf.eq(ticker)].sort_values("date").copy()
    history["target"] = history.log_rv_1m.shift(-1)
    history["target_date"] = history.date.shift(-1)
    history = history.dropna(subset=["log_rv_1m", "target"]).reset_index(drop=True)
    saved = d[d.model.eq("ols_1m") & d.etf.eq(ticker)].set_index("date")
    for i in range(60, len(history)):
        row = history.iloc[i]
        if row.date not in saved.index:
            continue
        s = saved.loc[row.date]
        X = np.column_stack([np.ones(i), history.log_rv_1m.iloc[:i].to_numpy()])
        y = history.target.iloc[:i].to_numpy()
        pred = np.array([1, row.log_rv_1m]) @ np.linalg.lstsq(X, y, rcond=None)[0]
        oof = np.full(i, np.nan)
        for tr, te in TimeSeriesSplit(n_splits=5).split(X):
            oof[te] = X[te] @ np.linalg.lstsq(X[tr], y[tr], rcond=None)[0]
        tested = np.isfinite(oof)
        sigma2 = np.var(y[tested] - oof[tested])
        assert history.target_date.iloc[i - 1] <= row.date < row.target_date
        assert row.target_date == s.target_date and np.isclose(row.target, s.actual_logRV)
        audit_rows.append(dict(etf=ticker, date=row.date, log_error=abs(pred - s.fc_log),
                               sigma2_error=abs(sigma2 - s.sigma2)))
        benchmark_rows.append(dict(etf=ticker, date=row.date, actual=row.target,
                                   ols_1m=s.fc_log, expanding_mean=y.mean(), random_walk=row.log_rv_1m))
ols_audit = pd.DataFrame(audit_rows)
assert len(ols_audit) == len(d[d.model.eq("ols_1m")])
assert ols_audit[["log_error", "sigma2_error"]].to_numpy().max() < 1e-10
benchmark = pd.DataFrame(benchmark_rows)
benchmark_losses = pd.DataFrame([
    dict(model=model, MSE_log=((benchmark.actual - benchmark[model]) ** 2).mean())
    for model in ["ols_1m", "expanding_mean", "random_walk"]
])
print(f"Independently rebuilt {len(ols_audit)} OLS forecasts, including sigma2.")
print(ols_audit[["log_error", "sigma2_error"]].max().to_string())
benchmark_losses

Independently rebuilt 1070 OLS forecasts, including sigma2.
log_error       6.217249e-15
sigma2_error    3.219647e-15


,model,MSE_log
0,ols_1m,0.522865
1,expanding_mean,1.034748
2,random_walk,0.602968


## Diagnostics: refit the five extreme macro forecasts

Recreate the builder's fitting and tuning procedure using only rows before each origin,
then decompose its log prediction into the intercept and standardized feature contributions.
A contribution is relative to the training feature mean. Correlated predictors mean this
is an algebraic model decomposition, not a causal attribution. These refits intentionally
retain the builder's internal CV procedure so the saved forecasts can be reproduced.


In [33]:
import warnings
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning

features = ["T10Y2Y", "DGS3MO", "BAA10Y", "CPIAUCSL", "T5YIE", "T5YIE_MINUS_T10YIE",
            "VIXCLS", "INDPRO", "UNRATE", "UMCSENT", "DCOILWTICO", "log_rv_1m"]
macro = pd.read_csv(
    MODELS_DIR.parent / "dataset/macro_variables/final_clean/final_macro_data.csv",
    index_col="date", parse_dates=True,
)
refit_rows, contribution_rows = [], []
# Reuse the same top five common-sample errors and RV data audited above.
for saved in top5.itertuples(index=False):
    history = rv_panel[rv_panel.etf.eq(saved.etf)].set_index("date").sort_index().copy()
    history["y_target"] = history.log_rv_1m.shift(-1)
    model_data = macro.join(history[["log_rv_1m", "y_target"]], how="inner")
    model_data = model_data[features + ["y_target"]].dropna()
    train = model_data.loc[model_data.index < saved.date]
    X, y = train[features].to_numpy(), train.y_target.to_numpy()
    scaler = StandardScaler()
    Z = scaler.fit_transform(X)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        cv = ElasticNetCV(
            l1_ratio=[.1, .3, .5, .7, .9, .95, .99, 1.],
            cv=TimeSeriesSplit(n_splits=5), max_iter=10000, random_state=0,
        ).fit(Z, y)
        model = ElasticNet(
            alpha=cv.alpha_, l1_ratio=cv.l1_ratio_, max_iter=10000, random_state=0,
        ).fit(Z, y)
        oof = np.full(len(y), np.nan)
        for tr, te in TimeSeriesSplit(n_splits=5).split(Z):
            oof[te] = clone(model).fit(Z[tr], y[tr]).predict(Z[te])
    z_test = scaler.transform(model_data.loc[[saved.date], features].to_numpy())
    pred = model.predict(z_test)[0]
    tested = np.isfinite(oof)
    sigma2 = np.var(y[tested] - oof[tested])
    refit_rows.append(dict(
        etf=saved.etf, date=saved.date, train_months=len(train),
        alpha=cv.alpha_, l1_ratio=cv.l1_ratio_, saved_fc_log=saved.fc_log,
        rebuilt_fc_log=pred, log_error=pred - saved.fc_log, sigma_error=sigma2 - saved.sigma2,
    ))
    for feature, value, standardized_x in zip(features, z_test[0] * model.coef_, z_test[0]):
        contribution_rows.append(dict(
            etf=saved.etf, date=saved.date, feature=feature,
            log_contribution=value, standardized_x=standardized_x,
        ))
extreme_refits = pd.DataFrame(refit_rows)
contributions = pd.DataFrame(contribution_rows)
assert extreme_refits[["log_error", "sigma_error"]].abs().to_numpy().max() < 1e-8
print("Reproduced all five extreme forecasts and their residual-variance corrections.")
print(contributions[contributions.feature.eq("VIXCLS")].to_string(index=False))
extreme_refits


Reproduced all five extreme forecasts and their residual-variance corrections.
etf       date feature  log_contribution  standardized_x
XLY 2008-10-31  VIXCLS          5.126728        9.520147
XLF 2008-10-31  VIXCLS          2.129736        9.520147
XLI 2008-10-31  VIXCLS          5.017184        9.520147
XLF 2009-03-31  VIXCLS          0.523214        2.177953
XLB 2008-10-31  VIXCLS          3.731073        9.520147


,etf,date,train_months,alpha,l1_ratio,saved_fc_log,rebuilt_fc_log,log_error,sigma_error
0,XLY,2008-10-31,68,0.048862,1.0,-0.528812,-0.528812,0.000000e+00,0.000000e+00
1,XLF,2008-10-31,68,0.020845,0.7,-0.929827,-0.929827,0.000000e+00,0.000000e+00
2,XLI,2008-10-31,68,0.044693,1.0,-1.069012,-1.069012,0.000000e+00,5.551115e-17
3,XLF,2009-03-31,73,0.010883,1.0,-1.343476,-1.343476,2.220446e-16,0.000000e+00
4,XLB,2008-10-31,68,0.033077,1.0,-1.584606,-1.584606,0.000000e+00,0.000000e+00


## Interpretation

- **Overall trend:** OLS and no-macro Elastic Net perform almost identically. Adding
  macro variables lowers log MSE by about 6.3% against either baseline, with little
  change in QLIKE. Gains are concentrated in 2010–2019 and 2021–2025 rather than crises.
- **Evidence:** DM does not establish a pooled macro improvement in log MSE
  (p ≈ 0.17) or QLIKE (p ≈ 0.47). Clark-West is strongly significant, but tests an
  adjusted differential; it does not establish significance of the raw gain.
- **Anomaly and explanation:** Full-macro level MSE is about ten times the baselines'
  (0.001075 versus 0.000107). Five errors in 2008–2009 account for 81.5% of it.
  Extreme inputs, especially VIX, drive large log forecasts that exponentiation and
  the variance correction amplify. QLIKE penalizes these over-forecasts much less
  sharply than squared level error, explaining the different rankings.
- **Checks:** Independent refits reproduce the extreme macro forecasts and all OLS
  forecasts. OLS target alignment and the saved variance correction pass the checks;
  the level-MSE anomaly reflects model extrapolation rather than a calculation error.
